# Sentinel-2 Composite Visualization & Verification

This notebook serves as the **Phase 1 Quality Gate** to load, verify, and visualize Sentinel-2 multispectral composite GeoTIFFs downloaded from Google Earth Engine.

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# Resolve path directory relative to notebook
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "sentinel"

print(f"[*] Project Root resolved to: {PROJECT_ROOT}")
print(f"[*] Data Folder resolved to: {DATA_DIR}")

ModuleNotFoundError: No module named 'rasterio'

## 1. Select Target Ingestion to Verify

In [ ]:
# Configure city and year to inspect
CITY = "Bengaluru"  # Bengaluru, Hyderabad, Pune
YEAR = "2019"       # 2019, 2026

target_folder = DATA_DIR / CITY / YEAR
image_path = target_folder / "image.tif"
metadata_path = target_folder / "metadata.json"

print(f"[*] Target Folder: {target_folder}")
print(f"[*] GeoTIFF Exists: {image_path.exists()}")
print(f"[*] Metadata Exists: {metadata_path.exists()}")

## 2. Load and Display Metadata parameters

In [ ]:
if metadata_path.exists():
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print(json.dumps(metadata, indent=4))
else:
    print(f"[!] Metadata file not found at: {metadata_path}")

## 3. Verify GeoTIFF Properties using Rasterio

In [ ]:
if image_path.exists():
    with rasterio.open(image_path) as src:
        print(f"=== GeoTIFF Header Diagnostics ===")
        print(f"[*] Width (pixels):  {src.width}")
        print(f"[*] Height (pixels): {src.height}")
        print(f"[*] Band Count:      {src.count}")
        print(f"[*] Coordinate Ref:  {src.crs}")
        print(f"[*] Pixel Resolution:{src.res}")
        print(f"[*] Affine Transform:\n{src.transform}")
        print(f"[*] Data Boundaries: {src.bounds}")
        print(f"[*] Bands Profile:   {src.descriptions}")
else:
    print(f"[!] GeoTIFF file not found at: {image_path}")

## 4. Display RGB Composite Visualizations

In [ ]:
if image_path.exists():
    with rasterio.open(image_path) as src:
        # Sentinel-2 raw band exports mapping order: 
        # Band 1 -> B2 (Blue)
        # Band 2 -> B3 (Green)
        # Band 3 -> B4 (Red)
        # Band 4 -> B8 (NIR)
        
        blue = src.read(1).astype(np.float32)
        green = src.read(2).astype(np.float32)
        red = src.read(3).astype(np.float32)
        
        # Combine bands into RGB array [H, W, 3]
        rgb = np.dstack([red, green, blue])
        
        # Sentinel-2 Surface Reflectance is uint16 (0-10000). 
        # We normalize reflectance bounds to max=3000 to stretch colors visually.
        rgb_normalized = np.clip(rgb / 3000.0, 0, 1)
        
        # Plotting
        plt.figure(figsize=(10, 10))
        plt.imshow(rgb_normalized)
        plt.title(f"Sentinel-2 RGB True Color Composite - {CITY} ({YEAR})\nScale stretched to 0-3000 DN", fontsize=14)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print(f"[!] GeoTIFF not available to visualize.")

## 5. View Stretched Near-Infrared (NIR) Band

Useful for vegetation assessments (NDVI precursors) since healthy green spaces reflect highly in the NIR wavelength spectrum.

In [ ]:
if image_path.exists():
    with rasterio.open(image_path) as src:
        # NIR is Band 4 (B8)
        nir = src.read(4).astype(np.float32)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(nir, cmap="Greens")
        plt.colorbar(label="Surface Reflectance DN")
        plt.title(f"Sentinel-2 NIR Band (B8) - {CITY} ({YEAR})", fontsize=14)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
